# CodeOrbit Internship Project 3 — Image Classification using a Pretrained Model
**Transfer learning with MobileNetV2 + Rock Paper Scissors image dataset**

Run each cell from top to bottom in Google Colab. Choose **Runtime → Change runtime type → T4 GPU** if available.

## 1. Install/import packages

In [ ]:
!pip -q install tensorflow-datasets

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 2. Load dataset
The Rock Paper Scissors dataset has three image categories. TFDS downloads it automatically.

In [ ]:
(ds_train, ds_test), info = tfds.load(
    "rock_paper_scissors",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

class_names = info.features["label"].names
print("Classes:", class_names)
print("Training examples:", info.splits["train"].num_examples)
print("Test examples:", info.splits["test"].num_examples)

## 3. Preview images

In [ ]:
plt.figure(figsize=(10, 6))
for i, (image, label) in enumerate(ds_train.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image)
    plt.title(class_names[int(label)])
    plt.axis("off")
plt.tight_layout()

## 4. Preprocess images

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = (ds_train
            .map(preprocess, num_parallel_calls=AUTOTUNE)
            .shuffle(1000)
            .batch(BATCH_SIZE)
            .prefetch(AUTOTUNE))

test_ds = (ds_test
           .map(preprocess, num_parallel_calls=AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(AUTOTUNE))

## 5. Load pretrained MobileNetV2

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

## 6. Train the classifier

In [ ]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

## 7. Plot accuracy and loss

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 8. Evaluate model

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print(f"Test loss: {test_loss:.4f}")

## 9. Predict a sample image

In [ ]:
sample_image, sample_label = next(iter(ds_test.take(1)))
display_image = tf.image.resize(sample_image, (IMG_SIZE, IMG_SIZE))
processed = tf.keras.applications.mobilenet_v2.preprocess_input(
    tf.cast(display_image, tf.float32)
)
prediction = model.predict(tf.expand_dims(processed, axis=0), verbose=0)[0]
predicted_class = class_names[int(np.argmax(prediction))]

plt.imshow(sample_image)
plt.axis("off")
plt.title(f"Predicted: {predicted_class} | Actual: {class_names[int(sample_label)]}")
plt.show()

print("Confidence:", f"{np.max(prediction) * 100:.2f}%")

## 10. Save the trained model

In [ ]:
model.save("codeorbit_image_classifier.keras")
print("Saved: codeorbit_image_classifier.keras")

## Project summary
- **Task:** classify hand gesture images into rock, paper, or scissors.
- **Pretrained model:** MobileNetV2 (ImageNet weights).
- **Approach:** transfer learning; freeze the pretrained feature extractor and train a small classification head.
- **Evaluation:** test accuracy, test loss, accuracy/loss plots, and a sample prediction.

Download the notebook using **File → Download → Download .ipynb**. After running, download the `.keras` model from Colab's Files panel if you want to include it in your submission.